# Week 7: The Adaptation Decision and a Production Dataset

## Part 1

Synthetic dataset for a personal assistant intent classification task.

In [12]:
import os, re, random, pathlib
from collections import Counter
random.seed(7)
# Synthetic intent classification data (template + slot fill). Slot lists are sized so every
# intent has well over N_TRAIN+N_EVAL unique texts after dedup.
INTENTS={'set_reminder':('remind me to {task} {time}','set a reminder to {task}','ping me to {task} {time}','reminder: {task} {time}'),
  'get_weather':('weather in {city}','will it rain in {city} {time}','how hot in {city}','forecast for {city} {time}'),
  'play_music':('play {x}','put on some {x}','queue up {x}','I want to hear {x}'),
  'send_message':('text {p} that {task}','message {p} about {task}','tell {p} {task}','let {p} know {task}'),
  'get_directions':('directions to {city}','navigate to {city} {time}','route to {city}','take me to {city}')}
SLOT={'task':['call mom','buy milk','submit the report','pay rent','book a flight','walk the dog','renew the license','email the client','restock coffee','back up the laptop'],
  'time':['tomorrow','at 5pm','tonight','this weekend','on monday','at noon','next week','before bed'],
  'city':['Denver','Tokyo','Lagos','Paris','Cairo','Oslo','Lima','Seoul','Madrid','Dublin','Nairobi','Sydney','Toronto','Berlin','Mumbai','Chicago','Rome','Bogota','Prague','Hanoi'],
  'x':['jazz','lofi','Adele','Miles Davis','reggaeton','classical','Fela Kuti','ambient','Bach','Beyonce','techno','bossa nova','Radiohead','Nina Simone','country','podcasts','synthwave','blues','Bob Marley','k-pop','metal','gospel','folk','Daft Punk'],
  'p':['Sam','my brother','the team','Priya','Dad','Marcus','my sister','the group']}
def fill(t): return re.sub(r'\{(\w+)\}', lambda m: random.choice(SLOT[m.group(1)]), t)
raw=[{'text':fill(random.choice(tmpl)),'label':lab} for lab,tmpl in INTENTS.items() for _ in range(140)]
# noise to clean: empty text, out-of-schema label, whitespace/case variants of existing rows
raw+=[{'text':'','label':'set_reminder'}]*4+[{'text':'hi','label':'unknown'}]*3
raw+=[{'text':'  '+r['text'].upper()+' ','label':r['label']} for r in random.sample(raw, 6)]
print('raw pool:', len(raw))

# Hand-written eval-only slice: phrasing outside the templates, borrowing vocabulary from a
# neighboring intent.
HARD=[
 # set_reminder: self-directed, but borrows message/weather/music vocabulary
 ('text me when it is time to walk the dog','set_reminder'),
 ('message me about the report tonight','set_reminder'),
 ("don't let me forget to buy milk",'set_reminder'),
 ('I keep forgetting to pay rent, nudge me on monday','set_reminder'),
 ('remind me to check the weather in Denver tomorrow','set_reminder'),
 ('remind me to play jazz at the party','set_reminder'),
 # send_message: addressed to another person, but borrows reminder/weather/music/directions vocabulary
 ('remind Priya to pay rent','send_message'),
 ('ping Dad to renew the license','send_message'),
 ('tell Sam to call mom tomorrow','send_message'),
 ('text Sam that I want to hear jazz','send_message'),
 ('tell the team the weather in Oslo is awful','send_message'),
 ('message Priya directions to Cairo','send_message'),
 ("text my brother that I'm heading to Lima",'send_message'),
 # get_weather: borrows travel vocabulary
 ('is it raining on the way to Oslo','get_weather'),
 ('should I bring an umbrella to Seoul','get_weather'),
 ('how hot is it along the route to Cairo','get_weather'),
 ('do I need a jacket in Berlin tonight','get_weather'),
 ('is Hanoi humid this weekend','get_weather'),
 # get_directions: borrows weather vocabulary or has no template verb
 ('best route to Tokyo avoiding the storm','get_directions'),
 ('how do I get to Seoul from here','get_directions'),
 ('get me to Oslo before noon','get_directions'),
 ('fastest way to Dublin in this rain','get_directions'),
 ('walking route to Madrid','get_directions'),
 # play_music: titles/phrases containing city or weather words
 ('play Paris by the Chainsmokers','play_music'),
 ('queue up Tokyo Drift','play_music'),
 ('play some rain sounds','play_music'),
 ('put on Dublin folk','play_music'),
 ('I want to hear Lagos Disco Party','play_music'),
 ('play the forecast playlist','play_music'),
]
print('hard slice:', len(HARD), dict(Counter(l for _,l in HARD)))

raw pool: 713
hard slice: 29 {'set_reminder': 6, 'send_message': 7, 'get_weather': 5, 'get_directions': 5, 'play_music': 6}


### Eval slices and labeling convention

The eval set has two slices. `template` is held out from the same generator as train (easy, near-duplicate phrasing). `ambiguous` is hand-written, outside the templates, and deliberately borrows vocabulary from a neighboring intent. Labels follow one convention: **the label is the action the speaker wants performed, and a request addressed to another person is `send_message` even if its content mentions a reminder, weather, music, or directions.** A self-directed "remind me" is `set_reminder`.

In [13]:
N_TRAIN, N_EVAL = 40, 20   # per label -> 200 train / 100 template-slice eval (+ the hand-written HARD slice)
VALID=list(INTENTS)        # a list, not a set: set iteration order varies per process and would change the shuffle/split between runs

# --- filtering + normalization + dedup, with a count for every drop reason
drops=Counter(); clean=[]
for r in raw:
    t=' '.join(r['text'].split())               # collapse whitespace
    if not t: drops['empty text']+=1
    elif r['label'] not in VALID: drops['label outside schema']+=1
    else: clean.append({'text':t,'label':r['label']})
seen=set(); dedup=[]
for r in clean:
    k=(r['text'].lower(), r['label'])           # case-insensitive
    if k in seen: drops['duplicate']+=1
    else: seen.add(k); dedup.append(r)
labels_by_text={}
for r in dedup: labels_by_text.setdefault(r['text'].lower(), set()).add(r['label'])
conflicts=[t for t,l in labels_by_text.items() if len(l)>1]
print(f'raw {len(raw)} -> clean {len(dedup)} | drops: {dict(drops)} | conflicting-label texts: {len(conflicts)}')

# --- stratified split
by={lab:[r for r in dedup if r['label']==lab] for lab in VALID}
train=[]; eval_=[]
for lab,rows in by.items():
    assert len(rows)>=N_TRAIN+N_EVAL, f'{lab}: only {len(rows)} unique rows'
    random.shuffle(rows); train+=rows[:N_TRAIN]; eval_+=rows[N_TRAIN:N_TRAIN+N_EVAL]
random.shuffle(train); random.shuffle(eval_)
for r in eval_: r['slice']='template'
eval_+=[{'text':t,'label':l,'slice':'ambiguous'} for t,l in HARD]   # appended after the shuffles so the template rows are unaffected
print('train',len(train),'eval',len(eval_), dict(Counter(r['slice'] for r in eval_)))

# --- validation
assert len(train)>=200 and len(eval_)>=50
assert not conflicts
tr_texts={r['text'].lower() for r in train}; ev_texts=[r['text'].lower() for r in eval_]
assert not tr_texts & set(ev_texts), 'train/eval leakage'
assert len(set(ev_texts))==len(ev_texts), 'duplicate eval text'
print('train/eval overlap: 0')
def ok(r, keys): return (set(r)==keys and isinstance(r['text'],str) and r['text']==r['text'].strip()
                         and '\n' not in r['text'] and r['label'] in VALID)
fmt_ok=all(ok(r,{'text','label'}) for r in train) and all(ok(r,{'text','label','slice'}) for r in eval_)
print('format check (keys, types, no stray whitespace/newlines, label in schema):', fmt_ok)
assert fmt_ok

# --- statistics
print('label balance  train:', dict(Counter(r['label'] for r in train)))
for s in ('template','ambiguous'):
    print(f'label balance  eval/{s}:', dict(Counter(r['label'] for r in eval_ if r['slice']==s)))
import tiktoken, statistics
os.environ['TIKTOKEN_CACHE_DIR']=str((pathlib.Path('.')/'.tiktoken_cache').resolve())
enc=tiktoken.get_encoding('cl100k_base')
for name,rows in (('train',train),('eval/template',[r for r in eval_ if r['slice']=='template']),('eval/ambiguous',[r for r in eval_ if r['slice']=='ambiguous'])):
    lens=sorted(len(enc.encode(r['text'])) for r in rows)
    print(f'token length {name}: min {lens[0]} median {statistics.median(lens)} mean {statistics.mean(lens):.1f} p95 {lens[int(.95*len(lens))]} max {lens[-1]}')
lens=[len(enc.encode(r['text'])) for r in train]
for n,c in sorted(Counter(lens).items()): print(f'  {n:>2} tok | {"#"*c} {c}')

raw 713 -> clean 449 | drops: {'empty text': 4, 'label outside schema': 3, 'duplicate': 257} | conflicting-label texts: 0
train 200 eval 129 {'template': 100, 'ambiguous': 29}
train/eval overlap: 0
format check (keys, types, no stray whitespace/newlines, label in schema): True
label balance  train: {'play_music': 40, 'send_message': 40, 'get_weather': 40, 'set_reminder': 40, 'get_directions': 40}
label balance  eval/template: {'send_message': 20, 'set_reminder': 20, 'play_music': 20, 'get_directions': 20, 'get_weather': 20}
label balance  eval/ambiguous: {'set_reminder': 6, 'send_message': 7, 'get_weather': 5, 'get_directions': 5, 'play_music': 6}
token length train: min 2 median 5.0 mean 5.5 p95 9 max 11
token length eval/template: min 2 median 5.0 mean 5.5 p95 9 max 10
token length eval/ambiguous: min 4 median 7 mean 7.2 p95 10 max 12
   2 tok | #### 4
   3 tok | ###################### 22
   4 tok | ##################################### 37
   5 tok | #################################

## Part 2: Adapt with dynamic few-shot

In [3]:
os.environ['HF_HOME']=str((pathlib.Path('.')/'.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
train_emb=emb.encode([r['text'] for r in train], convert_to_tensor=True, normalize_embeddings=True)
def few_shot(query, k=4):
    qe=emb.encode([query], convert_to_tensor=True, normalize_embeddings=True)
    sims=util.cos_sim(qe, train_emb).cpu().numpy()[0]
    return [train[i] for i in sims.argsort()[::-1][:k]]
ex=eval_[0]
print('eval input:', ex['text'], '(true:', ex['label'], ')')
print('retrieved few-shot examples:')
for s in few_shot(ex['text'], 4): print('  ', s['text'], '->', s['label'])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

eval input: tell my brother email the client (true: send_message )
retrieved few-shot examples:
   tell Dad email the client -> send_message
   remind me to email the client next week -> set_reminder
   tell Sam email the client -> send_message
   remind me to email the client tonight -> set_reminder


## Part 3 Evaluate

In [7]:
import time, json
API_OK=bool(os.environ.get('GEMINI_API_KEY'))
MODEL='gemini-3.1-flash-lite'
MIN_INTERVAL=60/14                  # free tier allows 15 requests/min; stay just under
CACHE=pathlib.Path('gemini_cache.json')   # completed calls survive a crash/rerun
cache=json.loads(CACHE.read_text()) if CACHE.exists() else {}
_last=0.0

if API_OK:
    from openai import OpenAI, RateLimitError
    client=OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')

def gemini_label(prompt, model=MODEL, retries=6):
    global _last
    key=f'{model}\n{prompt}'
    if key in cache: return cache[key]
    for attempt in range(retries):
        time.sleep(max(0, MIN_INTERVAL-(time.time()-_last)))   # client-side throttle
        _last=time.time()
        try:
            out=client.chat.completions.create(model=model, temperature=0,
                    messages=[{'role':'user','content':prompt}]).choices[0].message.content.strip()
        except RateLimitError as e:
            if 'PerDay' in str(e): raise                       # daily quota: retrying won't help
            m=re.search(r'retry in ([\d.]+)s', str(e))         # server-suggested delay, else backoff
            time.sleep((float(m.group(1)) if m else 2**attempt)+1)
            continue
        cache[key]=out; CACHE.write_text(json.dumps(cache))
        return out
    raise RuntimeError(f'still rate limited after {retries} retries')

def classify(text, shots=None):
    head='Classify the intent into one of: '+', '.join(sorted(VALID))+'.\n'   # sorted: stable prompt -> stable cache keys
    body=''.join(f'Text: {s["text"]}\nIntent: {s["label"]}\n' for s in (shots or []))
    return gemini_label(head+body+f'Text: {text}\nIntent:')

def parse(pred):
    # first schema label mentioned in the reply; tolerates markdown, case, "set reminder"
    p=re.sub(r'[\s\-]+','_',(pred or '').lower())
    hits=[(p.find(l),l) for l in VALID if l in p]
    return min(hits)[1] if hits else None

In [8]:
results=[]
if API_OK:
    # cold cache: ~2 calls per eval row at 14/min; a rerun after any failure resumes from gemini_cache.json
    for i,r in enumerate(eval_):
        shots=few_shot(r['text'])
        results.append({**r, 'shots':shots,
                        'zero':parse(classify(r['text'])),
                        'few':parse(classify(r['text'], shots))})
        if (i+1)%10==0: print(f'{i+1}/{len(eval_)}')
    for sl in (None,'template','ambiguous'):
        rows=[x for x in results if sl is None or x['slice']==sl]
        print(f'{sl or "all":<10} n={len(rows):<4}' + ''.join(
              f'{name}-shot {sum(x[name]==x["label"] for x in rows)/len(rows):.1%} (unparseable {sum(x[name] is None for x in rows)})   '
              for name in ('zero','few')))
else:
    print('[API-BLOCKED] set GEMINI_API_KEY to run the zero-shot vs dynamic-few-shot evaluation.')
    print('The dataset and the example selection above are real; only the scored eval needs the model.')

10/129
20/129
30/129
40/129
50/129
60/129
70/129
80/129
90/129
100/129
110/129
120/129
all        n=129 zero-shot 98.4% (unparseable 0)   few-shot 97.7% (unparseable 0)   
template   n=100 zero-shot 100.0% (unparseable 0)   few-shot 100.0% (unparseable 0)   
ambiguous  n=29  zero-shot 93.1% (unparseable 0)   few-shot 89.7% (unparseable 0)   


### Slice analysis
Accuracy per true label, then every eval item where few-shot was wrong and zero-shot was right, with the retrieved shots that were in the prompt.

In [9]:
if results:
    print(f'{"slice / label":<28}{"n":>4}{"zero":>8}{"few":>8}')
    for sl in ('template','ambiguous'):
        for lab in [None]+sorted(VALID):
            rows=[x for x in results if x['slice']==sl and (lab is None or x['label']==lab)]
            z,f=(sum(x[k]==x['label'] for x in rows)/len(rows) for k in ('zero','few'))
            name=f'{sl} / {lab or "ALL"}'
            print(f'{name:<28}{len(rows):>4}{z:>8.0%}{f:>8.0%}{"   <- few-shot worse" if f<z else ""}')
    worse=[x for x in results if x['zero']==x['label'] and x['few']!=x['label']]
    print(f'\nfew-shot wrong / zero-shot right: {len(worse)} of {len(results)}')
    for x in worse:
        print(f'\n[{x["slice"]}] "{x["text"]}"  true={x["label"]}  zero={x["zero"]}  few={x["few"]}')
        for s in x['shots']: print(f'    shot: {s["text"]} -> {s["label"]}')
else:
    print('no results (API-BLOCKED)')

slice / label                  n    zero     few
template / ALL               100    100%    100%
template / get_directions     20    100%    100%
template / get_weather        20    100%    100%
template / play_music         20    100%    100%
template / send_message       20    100%    100%
template / set_reminder       20    100%    100%
ambiguous / ALL               29     93%     90%   <- few-shot worse
ambiguous / get_directions     5    100%    100%
ambiguous / get_weather        5    100%    100%
ambiguous / play_music         6    100%    100%
ambiguous / send_message       7     86%     86%
ambiguous / set_reminder       6     83%     67%   <- few-shot worse

few-shot wrong / zero-shot right: 1 of 129

[ambiguous] "text me when it is time to walk the dog"  true=set_reminder  zero=set_reminder  few=send_message
    shot: ping me to walk the dog at noon -> set_reminder
    shot: reminder: walk the dog tomorrow -> set_reminder
    shot: reminder: walk the dog on monday -> set_re

I struggled to create examples where dynamic few-shot underperformed zero-shot and even this example is hard to defend as a misclassification given the purposeful ambiguity. It failed because of the borrowed vocabulary from the send_message intent despite the fact that the majority of shots were good examples of the true intent.

## Part 4 Memo

The retrieval selected few-shot and zero shot worked so similarly in these examples that it was difficult to produce underperforming results. These results suggest zero shot with a good prompt alone is enough for the task. Despite how effective prompting alone was, given that the domain is a personal assistant it still might make sense to fine-tune a model on this kind of intent classification for tool calling if the deployment demands it. If this is for something meant to run locally on a mobile device for example having a smaller lower latency model could be worth the cost of fine-tuning.